# Notebook 01A — Set Up SageMaker MLflow App for Team 02

**Project:** Explainable Diabetes Risk Screening  
**Module:** ITI113 Machine Learning & Operations Project  
**Team:** team02  
**Primary ownership:** MLOps / deployment / system governance

This notebook creates or reuses the team's **SageMaker MLflow App**, verifies the team tag and S3 artifact path, and performs a connectivity-only MLflow test.

> This setup notebook does **not** log fabricated model-performance results. Model metrics are logged only by the modelling and pipeline notebooks after actual execution.

## 0. Project alignment

The approved project uses the BRFSS 2015 `diabetes_012_health_indicators_BRFSS2015.csv` dataset and a derived binary target:

- `0` → no diabetes
- `{1, 2}` → prediabetes or diabetes

Model A is Logistic Regression and Model B is XGBoost. MLflow is used to track parameters, metrics, models, plots, data version, and pipeline artifacts.

Team ownership used for traceability:

| Member | Member ID | Primary focus |
|---|---:|---|
| Teh Tze Chee | s201 | MLOps, deployment, system governance |
| Fong Siang Yi | s202 | XGBoost, comparison, explainability |
| Djony Pamudji | s203 | Logistic Regression, baseline/error analysis |

## 1. Install / update required packages

In [1]:
# SageMaker Python SDK V3 + current AWS MLflow integration
%pip install -U boto3 botocore "sagemaker>=3,<4" mlflow sagemaker-mlflow


Note: you may need to restart the kernel to use updated packages.


## 2. Configuration

The team and project values below are fixed to Team 02. The execution role is obtained from the **current SageMaker environment** using the SageMaker Python SDK V3 helper API rather than hard-coding an AWS account ID or role ARN.

If a different team member runs this setup notebook, update only `STUDENT_ID`.


In [2]:
import json
import re
import time
import tempfile
from datetime import datetime, timezone
from pathlib import Path

import boto3
from botocore.exceptions import ClientError
from sagemaker.core.helper.session_helper import get_execution_role

REGION = "ap-southeast-1"
COURSE = "ITI113"
SEMESTER = "26S1"

TEAM_ID = "team02"
STUDENT_ID = "S203"
PROJECT_NAME = "diabetes-risk"

CLASS_BUCKET = "nyp-26s1-iti113"
TEAM_PREFIX = f"iti113/{TEAM_ID}"

MLFLOW_APP_NAME = f"iti113-{SEMESTER.lower()}-{TEAM_ID}-mlflow-app"
ARTIFACT_STORE_URI = f"s3://{CLASS_BUCKET}/{TEAM_PREFIX}/mlflow-app-artifacts/"
EXPERIMENT_NAME = f"{COURSE}/{TEAM_ID}/{PROJECT_NAME}"

SET_AS_ACCOUNT_DEFAULT = False
SET_AS_DEFAULT_FOR_EXISTING_DOMAINS = False

session = boto3.Session(region_name=REGION)
sts = session.client("sts")
sm = session.client("sagemaker")
s3 = session.client("s3")

caller_identity = sts.get_caller_identity()
ACCOUNT_ID = caller_identity["Account"]
CALLER_ARN = caller_identity["Arn"]

# SageMaker Python SDK V3:
# get_execution_role() is located in sagemaker.core.helper.session_helper.
ROLE_ARN = get_execution_role()

# Safety check only when the course role name explicitly contains a team number.
m = re.search(r"ITI113[-_]?Team0*(\d+)", ROLE_ARN, flags=re.IGNORECASE)
if m and int(m.group(1)) != 2:
    raise PermissionError(
        f"Current SageMaker execution role appears to belong to Team {m.group(1)}, "
        f"but this notebook is configured for {TEAM_ID}: {ROLE_ARN}"
    )

MLFLOW_APP_TAGS = [
    {"Key": "Course", "Value": COURSE},
    {"Key": "Semester", "Value": SEMESTER},
    {"Key": "TeamId", "Value": TEAM_ID},
    {"Key": "StudentId", "Value": STUDENT_ID},
    {"Key": "ProjectName", "Value": PROJECT_NAME},
    {"Key": "CreatedByNotebook", "Value": "01A_setup_sagemaker_mlflow_app"},
]

print("Account        :", ACCOUNT_ID)
print("Caller ARN     :", CALLER_ARN)
print("Execution role :", ROLE_ARN)
print("Region         :", REGION)
print("Team ID        :", TEAM_ID)
print("Student ID     :", STUDENT_ID)
print("Project        :", PROJECT_NAME)
print("MLflow App     :", MLFLOW_APP_NAME)
print("Artifact store :", ARTIFACT_STORE_URI)
print("Experiment     :", EXPERIMENT_NAME)


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Account        : 044528205969
Caller ARN     : arn:aws:sts::044528205969:assumed-role/SageMakerExecutionRole-ITI113-Team02/SageMaker
Execution role : arn:aws:iam::044528205969:role/SageMakerExecutionRole-ITI113-Team02
Region         : ap-southeast-1
Team ID        : team02
Student ID     : S203
Project        : diabetes-risk
MLflow App     : iti113-26s1-team02-mlflow-app
Artifact store : s3://nyp-26s1-iti113/iti113/team02/mlflow-app-artifacts/
Experiment     : ITI113/team02/diabetes-risk


## 3. Check SDK / API support

In [3]:
required_methods = [
    "create_mlflow_app",
    "list_mlflow_apps",
    "describe_mlflow_app",
    "create_presigned_mlflow_app_url",
    "add_tags",
    "list_tags",
]

missing = [method for method in required_methods if not hasattr(sm, method)]
if missing:
    raise RuntimeError(
        "The installed boto3 SageMaker client is missing these MLflow App APIs: "
        f"{missing}. Update boto3/botocore, restart the kernel, and rerun."
    )

print("Required SageMaker MLflow App APIs are available.")

Required SageMaker MLflow App APIs are available.


## 4. Verify the team MLflow artifact prefix is writable

In [4]:
from urllib.parse import urlparse

def parse_s3_uri(uri: str):
    parsed = urlparse(uri)
    if parsed.scheme != "s3":
        raise ValueError(f"Not an S3 URI: {uri}")
    return parsed.netloc, parsed.path.lstrip("/")

artifact_bucket, artifact_prefix = parse_s3_uri(ARTIFACT_STORE_URI)
if artifact_bucket != CLASS_BUCKET:
    raise ValueError("Artifact bucket differs from the configured ITI113 class bucket.")
if not artifact_prefix.startswith(f"{TEAM_PREFIX}/"):
    raise ValueError(
        f"Artifact prefix must remain under {TEAM_PREFIX}/; got {artifact_prefix}"
    )

test_key = f"{artifact_prefix.rstrip('/')}/_setup_checks/{STUDENT_ID}_{int(time.time())}.txt"
s3.put_object(
    Bucket=artifact_bucket,
    Key=test_key,
    Body=b"ITI113 Team02 MLflow artifact-store write test",
    ServerSideEncryption="AES256",
)
s3.delete_object(Bucket=artifact_bucket, Key=test_key)

print("S3 write/delete check passed:", ARTIFACT_STORE_URI)

S3 write/delete check passed: s3://nyp-26s1-iti113/iti113/team02/mlflow-app-artifacts/


## 5. Optional discovery of SageMaker Studio domain IDs

In [5]:
domain_ids = []
try:
    paginator = sm.get_paginator("list_domains")
    for page in paginator.paginate():
        for item in page.get("Domains", []):
            domain_ids.append(item["DomainId"])
    print("Studio domain IDs:", domain_ids or "None discovered")
except ClientError as e:
    print("[INFO] Domain discovery is not required for this project.")
    print(type(e).__name__, e)

Studio domain IDs: ['d-popmr5pqbh1n', 'd-wcuptup1pj6w', 'd-gpdrdk2w4dgw', 'd-5q0cdlsfisve', 'd-lsrccj8tewjf', 'd-oijl6tgx8os2', 'd-cydcdxkc4yot', 'd-tjsofl2bch3a', 'd-8nb3rzhhmygx', 'd-jgu4uwvdu9ir', 'd-ucuqj5x4jpp2', 'd-enscfq0pwdlh', 'd-3bkphr0cwggn', 'd-bigthb6rme9d', 'd-dryelcmwbiys', 'd-mbvdjlwsub3m', 'd-zcyz1ncfisrq', 'd-ragv9d2df9gy', 'd-ma4hqfpuzcku', 'd-iyisnrl8pcax', 'd-kfznavht8svt', 'd-x7nlty35iceg', 'd-zcdvxbv2u7ej', 'd-zjad5kjaiedi', 'd-crccom5725us', 'd-tjyhwxpu8ww7']


## 6. Create or reuse the Team 02 SageMaker MLflow App

In [6]:
def find_mlflow_app_by_name(name: str):
    paginator = sm.get_paginator("list_mlflow_apps")
    for page in paginator.paginate():
        for summary in page.get("Summaries", []):
            if (
                summary.get("Name") == name
                and summary.get("Status") not in {"Deleting", "Deleted"}
            ):
                return summary
    return None

def ensure_mlflow_app_tags(resource_arn: str, required_tags: list):
    required = {tag["Key"]: tag["Value"] for tag in required_tags}
    existing = {
        tag["Key"]: tag["Value"]
        for tag in sm.list_tags(ResourceArn=resource_arn).get("Tags", [])
    }
    updates = [
        {"Key": key, "Value": value}
        for key, value in required.items()
        if existing.get(key) != value
    ]
    if updates:
        sm.add_tags(ResourceArn=resource_arn, Tags=updates)

    final_tags = {
        tag["Key"]: tag["Value"]
        for tag in sm.list_tags(ResourceArn=resource_arn).get("Tags", [])
    }
    if final_tags.get("TeamId") != TEAM_ID:
        raise PermissionError(
            f"MLflow App TeamId tag is {final_tags.get('TeamId')!r}, "
            f"expected {TEAM_ID!r}."
        )
    return final_tags

existing = find_mlflow_app_by_name(MLFLOW_APP_NAME)

if existing:
    mlflow_app_arn = existing["Arn"]
    print("Reusing existing MLflow App:", mlflow_app_arn)
    final_tags = ensure_mlflow_app_tags(mlflow_app_arn, MLFLOW_APP_TAGS)
else:
    create_args = {
        "Name": MLFLOW_APP_NAME,
        "ArtifactStoreUri": ARTIFACT_STORE_URI,
        "RoleArn": ROLE_ARN,
        "ModelRegistrationMode": "AutoModelRegistrationDisabled",
        "Tags": MLFLOW_APP_TAGS,
    }
    if SET_AS_ACCOUNT_DEFAULT:
        create_args["AccountDefaultStatus"] = "ENABLED"
    if SET_AS_DEFAULT_FOR_EXISTING_DOMAINS and domain_ids:
        create_args["DefaultDomainIdList"] = domain_ids

    response = sm.create_mlflow_app(**create_args)
    mlflow_app_arn = response["Arn"]
    print("Created MLflow App:", mlflow_app_arn)
    final_tags = ensure_mlflow_app_tags(mlflow_app_arn, MLFLOW_APP_TAGS)

print("Verified MLflow App tags:")
print(json.dumps(final_tags, indent=2))

Reusing existing MLflow App: arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-YHNC33FF6GPZ
Verified MLflow App tags:
{
  "sagemaker:user-profile-arn": "arn:aws:sagemaker:ap-southeast-1:044528205969:user-profile/d-zjad5kjaiedi/team02-s203",
  "Semester": "26S1",
  "sagemaker:domain-arn": "arn:aws:sagemaker:ap-southeast-1:044528205969:domain/d-zjad5kjaiedi",
  "ProjectName": "diabetes-risk",
  "sagemaker:space-arn": "arn:aws:sagemaker:ap-southeast-1:044528205969:space/d-zjad5kjaiedi/project-space-team02-s203",
  "Course": "ITI113",
  "TeamId": "team02",
  "CreatedByNotebook": "01A_setup_sagemaker_mlflow_app",
  "StudentId": "S203"
}


In [7]:
def wait_for_mlflow_app(arn: str, timeout_seconds: int = 900, poll_seconds: int = 20):
    """
    Wait until the SageMaker MLflow App reaches a usable state.

    Current MLflow App API statuses:
    Creating, Created, CreateFailed, Updating, Updated,
    UpdateFailed, Deleting, DeleteFailed, Deleted.
    """
    ready_statuses = {"Created", "Updated"}
    failure_statuses = {"CreateFailed", "UpdateFailed", "DeleteFailed", "Deleted"}

    deadline = time.time() + timeout_seconds
    while time.time() < deadline:
        desc = sm.describe_mlflow_app(Arn=arn)
        status = desc.get("Status")
        print("MLflow App status:", status)

        if status in ready_statuses:
            return desc

        if status in failure_statuses:
            raise RuntimeError(
                "MLflow App entered a non-usable state:\n"
                + json.dumps(desc, indent=2, default=str)
            )

        if status == "Deleting":
            raise RuntimeError("MLflow App is being deleted and cannot be used.")

        time.sleep(poll_seconds)

    raise TimeoutError(
        f"MLflow App did not become ready within {timeout_seconds}s."
    )

mlflow_app_description = wait_for_mlflow_app(mlflow_app_arn)

print("MLflow version :", mlflow_app_description.get("MlflowVersion"))
print("MLflow status  :", mlflow_app_description.get("Status"))


MLflow App status: Created
MLflow version : 3.10.1
MLflow status  : Created


## 7. Generate a presigned MLflow UI URL

In [9]:
url_response = sm.create_presigned_mlflow_app_url(Arn=mlflow_app_arn)
mlflow_ui_url = (
    url_response.get("AuthorizedUrl")
    or url_response.get("Url")
    or url_response.get("PresignedUrl")
)

print("Presigned MLflow UI URL:")
print(mlflow_ui_url)

Presigned MLflow UI URL:
https://app-YHNC33FF6GPZ.mlflow.sagemaker.ap-southeast-1.app.aws/auth?authToken=eyJhbGciOiJIUzI1NiJ9.eyJhdXRoVG9rZW5JZCI6IlhHR1UyNyIsImZhc0NyZWRlbnRpYWxzIjoiQWdWNHlpQXVnbTFiWnFIQTdLRjRYR3lNVTZFeUFsZUQvWXhwZmJyQVB1dUhObFlBWHdBQkFCVmhkM010WTNKNWNIUnZMWEIxWW14cFl5MXJaWGtBUkVFeVMwUlNVbVZ4Wm10d2VITkJWa1ZFYkRoRFZWSm9RMmxXYldSdmJYTlJkVlpaYVVaMFJsVmpNWGQxYkhwaFMwUmlOamszU0hGNVV6VTJZazE1VW5jdlFUMDlBQUVBQjJGM2N5MXJiWE1BVUdGeWJqcGhkM002YTIxek9tRndMWE52ZFhSb1pXRnpkQzB4T2pNNU5qa3hNemN6TnpJMU5EcHJaWGt2WVRBNU1XRmhNRE10TnprMU5TMDBaakF5TFdJMVpHWXRaVE5oTlRNd1pXSmlaVGcxQUxnQkFnRUFlT0thVkkrUUdqak5TNEo0TUhCNk91SlA3UGFLdlRHSG9tY2kveDlrZTJiekFWSFpPN3k3Ujk2SGVTQzBsQmdGdzdVQUFBQitNSHdHQ1NxR1NJYjNEUUVIQnFCdk1HMENBUUF3YUFZSktvWklodmNOQVFjQk1CNEdDV0NHU0FGbEF3UUJMakFSQkF4QnV5b2gzb2hLVFZ2TVR0NENBUkNBT3k1ZVhLWWlJWjQ3eXVTYlI2L2Irbi82WFUvbHFLcExEUE94R1h6MGVWWWoyMHBGWGN1eXBqSktNQk1VbndOL2gvQzltVXlXODZLclg1Ri9BZ0FBRUFBUkZhWEs2OXByOHJ1NkpGZkltZkp0dWU3YVRTR3Rpb1M4ckVKa1NHajgyZ2FmejdlSk9xSWZweGY3L3

## 8. Connectivity-only MLflow logging test

The MLflow client connects using the SageMaker MLflow App ARN. If AWS reports a client/server MLflow version mismatch, install the MLflow client version matching the `MlflowVersion` printed in the previous step before continuing.


In [10]:
import mlflow

mlflow.set_tracking_uri(mlflow_app_arn)
mlflow.set_experiment(EXPERIMENT_NAME)

run_name = f"{TEAM_ID}_{STUDENT_ID}_mlflow_setup_check"

with mlflow.start_run(run_name=run_name) as run:
    mlflow.set_tags({
        "course": COURSE,
        "semester": SEMESTER,
        "team_id": TEAM_ID,
        "student_id": STUDENT_ID,
        "project_name": PROJECT_NAME,
        "tracking_backend": "sagemaker_mlflow_app",
        "purpose": "connectivity_only",
        "not_model_evidence": "true",
    })
    mlflow.log_params({
        "team_id": TEAM_ID,
        "student_id": STUDENT_ID,
        "region": REGION,
        "artifact_store_uri": ARTIFACT_STORE_URI,
    })
    mlflow.log_metric("setup_connectivity_check", 1.0)

    summary = {
        "message": "SageMaker MLflow App connectivity check succeeded.",
        "team_id": TEAM_ID,
        "student_id": STUDENT_ID,
        "project_name": PROJECT_NAME,
        "mlflow_app_arn": mlflow_app_arn,
        "experiment_name": EXPERIMENT_NAME,
        "run_id": run.info.run_id,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "note": "This is not a model-performance run.",
    }

    with tempfile.TemporaryDirectory() as tmpdir:
        p = Path(tmpdir) / "mlflow_setup_summary.json"
        p.write_text(json.dumps(summary, indent=2), encoding="utf-8")
        mlflow.log_artifact(str(p), artifact_path="setup_check")

    run_id = run.info.run_id

print("Connectivity check logged.")
print("Run ID:", run_id)

🏃 View run team02_S203_mlflow_setup_check at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/2/runs/4f73d20ac07c433389a1803d850168e6
🧪 View experiment at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/2
Connectivity check logged.
Run ID: 4f73d20ac07c433389a1803d850168e6


## 9. Save team MLflow configuration for Notebooks 02 and 03

In [11]:
config = {
    "REGION": REGION,
    "MLFLOW_APP_ARN": mlflow_app_arn,
    "EXPERIMENT_NAME": EXPERIMENT_NAME,
    "TEAM_ID": TEAM_ID,
    "STUDENT_ID": STUDENT_ID,
    "PROJECT_NAME": PROJECT_NAME,
    "ARTIFACT_STORE_URI": ARTIFACT_STORE_URI,
    "MLFLOW_APP_NAME": MLFLOW_APP_NAME,
    "MLFLOW_APP_TAGS": MLFLOW_APP_TAGS,
}

team_file = Path(f"mlflow_app_config_{TEAM_ID}.json")
student_file = Path(f"mlflow_app_config_{TEAM_ID}_{STUDENT_ID}.json")

for path in [team_file, student_file]:
    path.write_text(json.dumps(config, indent=2), encoding="utf-8")
    print("Saved:", path)

Saved: mlflow_app_config_team02.json
Saved: mlflow_app_config_team02_S203.json


## 10. Handoff

Notebook 02 should load `mlflow_app_config_team02.json` and verify that the MLflow App has `TeamId=team02` before logging any experiment.

No MLflow App ARN is hard-coded into the downstream notebooks. This prevents accidental cross-team logging.